# KV-compression risk pilot on Kaggle

This notebook runs the preregistered inference-only gate for the question: **is compression failure a stable, problem-specific property that could be predicted, or mostly perturbation noise and ordinary difficulty?**

It first screens GSM8K, MATH-500, and AIME 2024 using full-cache greedy accuracy. It then runs full cache, an exact full-cache repeat, and four generated-cache retention budgets on 150 disjoint questions. A small stochastic check estimates the sampling-noise floor. No predictor project should begin unless the final gates pass.

Enable a Kaggle **T4 or newer GPU** and Internet. Use **Save Version → Save & Run All** for the long run. Kaggle continues the saved run after you close the browser.

## 1. Configure

On the first run leave `RESUME_INPUT` empty. If a session reaches exit code 42, save its output as a Kaggle dataset, attach that dataset to the next notebook run, and either leave `RESUME_INPUT` empty for auto-discovery or set it to the attached `outputs/kv_compression_risk_pilot` directory.

Before the final Save & Run All, replace `RUN_COMMIT = "main"` with the immutable commit printed by setup.

In [ ]:
REPO_URL = "https://github.com/0x0shephard/latent-reasoning.git"
RUN_COMMIT = "main"
REPO_DIR = "/kaggle/working/latent-reasoning"

RESUME_INPUT = ""  # Optional attached outputs/kv_compression_risk_pilot directory.
RUN_SMOKE = True
RUN_DATASET_SCREEN = True
RUN_PRIMARY_PILOT = True
RUN_STOCHASTIC_CHECK = True
RUN_ANALYSIS = True

MAX_STAGE_SECONDS = 28800  # No individual stage receives more than 8 h.
MAX_SESSION_SECONDS = 36000  # Reserve time to export before Kaggle's hard limit.
EXPORT_RESERVE_SECONDS = 1800
UPLOAD_AS_KAGGLE_DATASET = False
KAGGLE_DATASET_HANDLE = "jonraza15/kv-compression-risk-pilot"

CONFIG = "configs/kv_risk_pilot.yaml"
OUTPUT_RELATIVE = "outputs/kv_compression_risk_pilot"
REPORT_RELATIVE = "reports/kv_compression_risk_pilot"
LOG_RELATIVE = "logs/kv_compression_risk_pilot"

## 2. Clone pinned code and install the pilot environment

In [ ]:
import datetime
import hashlib
import json
import os
import pathlib
import shutil
import subprocess
import sys
import time

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("Hugging Face authentication: Kaggle secret loaded")
except Exception:
    print("Hugging Face authentication: public access")

repo = pathlib.Path(REPO_DIR)
if not (repo / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
target = f"origin/{RUN_COMMIT}" if RUN_COMMIT == "main" else RUN_COMMIT
subprocess.run(["git", "-C", REPO_DIR, "checkout", "--detach", target], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements-kv-risk-pilot.txt")],
    check=True,
)
os.chdir(REPO_DIR)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Checked out:", commit)
if RUN_COMMIT == "main":
    print("PIN RUN_COMMIT BEFORE THE FINAL RUN:", commit)

## 3. Verify GPU and code contract

In [ ]:
import torch

assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator"
major, minor = torch.cuda.get_device_capability(0)
gpu_name = torch.cuda.get_device_name(0)
print("Torch:", torch.__version__, "GPU:", gpu_name, "capability:", (major, minor))
assert major >= 7, "Use a T4 or newer GPU. Current Kaggle PyTorch does not support the P100 reliably."

subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_kv_risk_cache.py", "tests/test_kv_risk_pilot.py", "tests/test_kaggle_kv_risk_pilot_notebook.py"],
    cwd=REPO_DIR,
    check=True,
)

## 4. Restore a previous partial run if attached

The output contains only compact JSON records and reports, not model weights. Every record is written atomically and has an identity hash, so repeating this notebook skips completed examples and rejects incompatible settings.

In [ ]:
OUTPUT_ROOT = repo / OUTPUT_RELATIVE
REPORT_ROOT = repo / REPORT_RELATIVE
LOG_ROOT = repo / LOG_RELATIVE
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
LOG_ROOT.mkdir(parents=True, exist_ok=True)

def discover_resume_root():
    if RESUME_INPUT:
        candidate = pathlib.Path(RESUME_INPUT)
        assert candidate.is_dir(), f"RESUME_INPUT does not exist: {candidate}"
        return candidate
    candidates = []
    for selection in pathlib.Path("/kaggle/input").rglob("kv_compression_risk_pilot/screen/dataset_selection.json"):
        candidates.append(selection.parent.parent)
    for manifest in pathlib.Path("/kaggle/input").rglob("kv_compression_risk_pilot/pilot/run_manifest.json"):
        candidates.append(manifest.parent.parent)
    unique = sorted({path.resolve() for path in candidates}, key=lambda path: (len(str(path)), str(path)))
    if len(unique) > 1:
        raise RuntimeError(f"Multiple resume trees found. Set RESUME_INPUT explicitly: {unique}")
    return unique[0] if unique else None

resume_root = discover_resume_root()
if resume_root is None:
    print("No prior pilot attached; starting a new durable run")
else:
    print("Restoring:", resume_root)
    shutil.copytree(resume_root, OUTPUT_ROOT, dirs_exist_ok=True)
    print("Restored to:", OUTPUT_ROOT)

## 5. Helpers and bounded smoke test

The smoke test exercises all cache conditions on two short MATH-500 examples. It is isolated from the preregistered output.

In [ ]:
SESSION_STARTED = time.monotonic()
SESSION_NEEDS_RESUME = False
SCREEN_REJECTED = False

def run_logged(command, log_name, allowed=(0, 42)):
    log_path = LOG_ROOT / log_name
    print("Starting:", " ".join(map(str, command)))
    print("Persistent log:", log_path)
    with log_path.open("a", encoding="utf-8", buffering=1) as log:
        log.write(f"\n=== {datetime.datetime.now(datetime.timezone.utc).isoformat()} {' '.join(map(str, command))} ===\n")
        process = subprocess.Popen(
            command,
            cwd=REPO_DIR,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        code = process.wait()
        log.flush()
    print("Exit code:", code)
    if code not in allowed:
        raise subprocess.CalledProcessError(code, command)
    return code

def stage_command(stage, output_root=OUTPUT_ROOT, extra=None):
    elapsed = time.monotonic() - SESSION_STARTED
    remaining = MAX_SESSION_SECONDS - EXPORT_RESERVE_SECONDS - elapsed
    budget = max(600.0, min(float(MAX_STAGE_SECONDS), remaining))
    return [
        sys.executable, "-u", "scripts/run_kv_risk_pilot.py",
        "--config", CONFIG,
        "--output-dir", str(output_root),
        "--stage", stage,
        "--device", "cuda",
        "--max-seconds", str(budget),
        *(extra or []),
    ]

if RUN_SMOKE:
    smoke_root = repo / "outputs/kv_compression_risk_pilot_smoke"
    if smoke_root.exists():
        shutil.rmtree(smoke_root)
    run_logged(
        stage_command("screen", smoke_root, ["--screen-examples", "2", "--max-new-tokens", "16", "--allow-selection-failure"]),
        "smoke_screen.log",
        allowed=(0,),
    )
    run_logged(
        stage_command("pilot", smoke_root, ["--dataset-override", "math500", "--pilot-examples", "2", "--max-new-tokens", "16"]),
        "smoke_pilot.log",
        allowed=(0,),
    )
    print("Smoke test passed")
else:
    print("Smoke test skipped")

## 6. Preregistered full-cache dataset screen

A dataset qualifies only if screen accuracy is 60–85%, median reasoning length is at least 512 generated tokens, and 150 disjoint examples remain. If none qualifies, the notebook stops the project rather than silently choosing a convenient dataset.

In [ ]:
if RUN_DATASET_SCREEN and not SESSION_NEEDS_RESUME:
    code = run_logged(stage_command("screen"), "screen.log", allowed=(0, 3, 42))
    SESSION_NEEDS_RESUME = code == 42
    SCREEN_REJECTED = code == 3
else:
    print("Dataset screen skipped")

selection_path = OUTPUT_ROOT / "screen/dataset_selection.json"
if selection_path.is_file():
    selection = json.loads(selection_path.read_text())
    print(json.dumps(selection, indent=2))
if SCREEN_REJECTED:
    print("PIVOT: no candidate passed the preregistered dataset gate")

## 7. Deterministic 150-question budget sweep

In [ ]:
if RUN_PRIMARY_PILOT and not SESSION_NEEDS_RESUME and not SCREEN_REJECTED:
    code = run_logged(stage_command("pilot"), "pilot.log", allowed=(0, 42))
    SESSION_NEEDS_RESUME = code == 42
else:
    print("Primary pilot skipped for this session")

## 8. Three-seed stochastic noise-floor check

In [ ]:
if RUN_STOCHASTIC_CHECK and not SESSION_NEEDS_RESUME and not SCREEN_REJECTED:
    code = run_logged(stage_command("stochastic"), "stochastic.log", allowed=(0, 42))
    SESSION_NEEDS_RESUME = code == 42
else:
    print("Stochastic check skipped for this session")

## 9. Apply the preregistered decision rule

In [ ]:
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
report_path = REPORT_ROOT / "kv_compression_risk_pilot.json"
if RUN_ANALYSIS and not SESSION_NEEDS_RESUME and not SCREEN_REJECTED:
    run_logged(
        [
            sys.executable, "-u", "scripts/analyze_kv_risk_pilot.py",
            "--config", CONFIG,
            "--statistics", str(OUTPUT_ROOT),
            "--output", str(report_path),
        ],
        "analysis.log",
        allowed=(0,),
    )
    from IPython.display import Image, Markdown, display
    display(Markdown(report_path.with_suffix(".md").read_text()))
    display(Image(filename=str(report_path.with_suffix(".png"))))
else:
    print("Analysis deferred until all stages are complete")

## 10. Export durable output

If `SESSION_NEEDS_RESUME` is true, this export is still valid. Save the notebook version with outputs enabled, optionally create a Kaggle dataset from the export, attach it to the next run, and rerun all cells.

In [ ]:
EXPORT_ROOT = pathlib.Path("/kaggle/working/kv_compression_risk_pilot_export")
if EXPORT_ROOT.exists():
    shutil.rmtree(EXPORT_ROOT)
export_repo = EXPORT_ROOT / "latent-reasoning"
for source, relative in (
    (OUTPUT_ROOT, OUTPUT_RELATIVE),
    (REPORT_ROOT, REPORT_RELATIVE),
    (LOG_ROOT, LOG_RELATIVE),
):
    if source.exists():
        shutil.copytree(source, export_repo / relative, dirs_exist_ok=True)

important = sorted(
    path for path in export_repo.rglob("*")
    if path.is_file() and path.suffix in {".json", ".jsonl", ".md", ".png"}
)
checksums = []
for path in important:
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    checksums.append(f"{digest}  {path.relative_to(EXPORT_ROOT)}")
(EXPORT_ROOT / "SHA256SUMS.txt").write_text("\n".join(checksums) + "\n")

print("Export:", EXPORT_ROOT)
print("Session needs resume:", SESSION_NEEDS_RESUME)
print("Screen rejected:", SCREEN_REJECTED)
print("Files:", sum(1 for path in EXPORT_ROOT.rglob("*") if path.is_file()))
if SESSION_NEEDS_RESUME:
    print("Save this version with outputs enabled, attach it to a new run, and rerun all cells.")
elif SCREEN_REJECTED:
    print("The preregistered screen rejected all candidates. Do not force a primary dataset.")
else:
    print("Pilot execution reached a terminal state. Inspect the decision report above.")

In [ ]:
if UPLOAD_AS_KAGGLE_DATASET:
    import kagglehub
    kagglehub.dataset_upload(
        KAGGLE_DATASET_HANDLE,
        str(EXPORT_ROOT),
        version_notes=f"KV compression risk pilot at commit {commit}; resume={SESSION_NEEDS_RESUME}",
    )
    print("Dataset upload complete:", KAGGLE_DATASET_HANDLE)
else:
    print("Dataset upload disabled. Save Version with outputs enabled.")